# ASL Pose Recognition - 90% Accuracy Training (Colab)

## Instructions:
1. Upload these files to Colab:
   - `processed_data/X_pose_seq.npy` (1.6GB - main features)
   - `processed_data/y_pose_seq.npy` (labels)
   - `processed_data/group_ids.npy` (group IDs for split)
   - `processed_data/labels.json` (word names)
2. Run all cells
3. Download `asl_pose_lstm_90_best.keras` (~5MB)
4. Copy back to local project
5. Test with `python diagnose_overfitting.py` → expect 90%+ val acc

**GPU recommended** (Runtime → Change runtime type → GPU)

In [ ]:
# Install dependencies
!pip install -q tensorflow==2.15.0 scikit-learn joblib h5py

In [ ]:
import os
import numpy as np
import joblib
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import GroupShuffleSplit
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import SparseCategoricalCrossentropy

print('TF version:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)

In [ ]:
# Load data
print('Loading data...')
X = np.load('processed_data/X_pose_seq.npy')
y = np.load('processed_data/y_pose_seq.npy')
groups = np.load('processed_data/group_ids.npy')

print(f'Samples: {X.shape}, Classes: {len(np.unique(y))}')

# Group-aware split: 70/15/15
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, temp_idx = next(gss.split(X, y, groups))

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss2.split(temp_idx, y[temp_idx], groups[temp_idx]))

X_train = X[train_idx]
X_val = X[temp_idx[val_idx]]
X_test = X[temp_idx[test_idx]]
y_train = y[train_idx]
y_val = y[temp_idx[val_idx]]
y_test = y[temp_idx[test_idx]]

print(f'Train: {X_train.shape}')
print(f'Val:   {X_val.shape}')
print(f'Test:  {X_test.shape}')

In [ ]:
# Normalize (fit scaler on train only)
scaler = StandardScaler()
SEQ_LEN = 13
FEATURES = 1280

X_train_flat = X_train.reshape(-1, FEATURES)
scaler.fit(X_train_flat)
X_train_flat = scaler.transform(X_train_flat)
X_train = X_train_flat.reshape(X_train.shape)

# Apply to val/test
X_val_flat = X_val.reshape(-1, FEATURES)
X_val = scaler.transform(X_val_flat).reshape(X_val.shape)

X_test_flat = X_test.reshape(-1, FEATURES)
X_test = scaler.transform(X_test_flat).reshape(X_test.shape)

# Save scaler for app
joblib.dump(scaler, 'feature_scaler.pkl')
print('Scaler saved')

In [ ]:
# Class weights & model config
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = {i: class_weights[i] for i in range(len(class_weights))}
num_classes = len(np.unique(y_train))

print(f'Classes: {num_classes}, weights sample: {dict(list(class_weights_dict.items())[:5])}')

In [ ]:
# 90% SOTA Model: Transformer Encoder + BiLSTM
inputs = tf.keras.Input(shape=(SEQ_LEN, FEATURES))

# Transformer Block 1
attn_out1 = tf.keras.layers.MultiHeadAttention(num_heads=8, key_dim=64)(inputs, inputs)
x1 = tf.keras.layers.LayerNormalization()(inputs + attn_out1)

# Transformer Block 2
attn_out2 = tf.keras.layers.MultiHeadAttention(num_heads=8, key_dim=64)(x1, x1)
x = tf.keras.layers.LayerNormalization()(x1 + attn_out2)

# BiLSTM
x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128, return_sequences=True, dropout=0.2))(x)
x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, dropout=0.2))(x)

# Classifier
x = tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
x = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=0.001, weight_decay=0.01),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Callbacks for 90% accuracy
callbacks = [
    tf.keras.callbacks.ModelCheckpoint('asl_pose_lstm_90_best.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

# Train
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val),
    class_weight=class_weights_dict,
    callbacks=callbacks,
    verbose=1
)

# Final scores
best_val = max(history.history['val_accuracy'])
print(f'\n🎯 BEST VAL ACCURACY: {best_val*100:.2f}%')
print('Model saved: asl_pose_lstm_90_best.keras')

# Save test data for local evaluation
np.save('X_test.npy', X_test)
np.save('y_test.npy', y_test)
print('Test data saved')

## Download Results
1. Download `asl_pose_lstm_90_best.keras`
2. Download `feature_scaler.pkl` 
3. Copy to local `Smart_ASL_Project/`
4. Test locally: `python diagnose_overfitting.py`
5. Run app: `python app.py`